# Feature Transformation: Categories, Explanations, and Use Cases

Feature transformation converts raw data into a format that is more suitable for building machine learning models. Below is a detailed explanation of each main category of feature transformation, along with when they are best used.

## 1. Feature Scaling

**Explanation:**  
Feature scaling is a **critical preprocessing step** in machine learning that adjusts the range or distribution of numerical features. Its core purpose is to **standardize independent variables** so they contribute equally to model training, preventing features with larger magnitudes from dominating those with smaller ranges. Feature scaling adjusts the range or distribution of *numerical features*, ensuring all features contribute equally to the analysis.

**Common Methods:**
- **Min-Max Scaling:** Transforms features to a fixed range (often 0–1).
- **Standardization (Z-score):** Centers features at mean 0 with standard deviation 1.
- **Robust Scaler:** Scales based on quantiles, robust to outliers. it uses **median and interquartile range (IQR)** to resist outliers. 
---

**Key Considerations When Scaling**  
| **Factor**               | **Min-Max**       | **Standardization** | **Robust Scaler**    |
|--------------------------|-------------------|---------------------|----------------------|
| **Outlier Sensitivity**  | High              | Moderate            | **None**             |
| **Output Range**         | Fixed (e.g., 0–1) | Unbounded           | Unbounded            |
| **Preserves Distribution**| Yes               | Shape only          | No                   |
| **Best For**             | Bounded algorithms| Gaussian-like data  | **Outlier-rich data**|

---

**Step-by-Step Implementation**  
1. **Split Data First**:  
   - **Never scale before splitting** train/test data to avoid data leakage.  
2. **Fit on Training Data**:  
   - Compute parameters (e.g., min/max, mean/std, median/IQR) **only on training set**.  
3. **Transform Test Data**:  
   - Apply scaling using training parameters.
   
**Applications:**
   - **Example (Supply Chain)**:  
     - Raw Data: `delivery_distance_km` (1-1000), `package_weight_kg` (0.1-500)  
     - Scaling: Min-Max scaling → both [0,1] range

**When to Use:**
- **Min-Max**:  
  - Neural networks (inputs require 0–1).  
  - Clustering (e.g., K-means).  
- **Standardization**:  
  - Linear models (regression, SVM).  
  - Dimensionality reduction (PCA).  
- **Robust Scaler**:  
  - Real-world data with outliers.  
  - Non-parametric models (e.g., decision trees benefit marginally).  

> **Tree-based algorithms** (Random Forest, XGBoost) **do not require scaling** since they split data independently of feature magnitude.  

**Common Pitfalls**  
- **Data Leakage**: Scaling before train-test split contaminates test data.  
- **Misinterpreting Bounds**: Standardized values can exceed [−3, 3] (e.g., extreme values).  
- **Categorical Features**: Never scale one-hot encoded/dummy variables.

In [6]:
#Min-Max Scaling:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.DataFrame({'salary': [20000, 50000, 80000]})
scaler = MinMaxScaler()
df['salary_scaled'] = scaler.fit_transform(df[['salary']])
print(df)

   salary  salary_scaled
0   20000            0.0
1   50000            0.5
2   80000            1.0


In [8]:
#Standard Scaling:

from sklearn.preprocessing import StandardScaler

data = [[1, 20000], [2, 50000], [3, 80000]]
scaler = StandardScaler()
scaled = scaler.fit_transform(data)
print(scaled)

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


## 2. Encoding Categorical Data

**Explanation:**  
Machine learning algorithms (regression, neural networks, SVM, etc.) require **numerical input**. Categorical data (text labels) must be converted to numbers while preserving their meaning. Improper encoding introduces false ordinal relationships (e.g., "Red"=1 vs "Blue"=2 implying "Blue" > "Red"). This transforms non-numeric (categorical) variables into a numeric format suitable for model consumption.

**Common Techniques:**
- **Label Encoding:** Assigns integer values to categories; works best for ordinal variables.  Example: ["Low", "Medium", "High"] → [0, 1, 2].  
- **One-Hot Encoding:** Creates binary columns for each category; ideal for nominal (unordered) features.
  Example: "Color" with values ["Red", "Blue"] becomes:  
  | **Color_Red** | **Color_Blue** |  
  |---------------|----------------|  
  | 1             | 0              |  
  | 0             | 1              |  
- **Ordinal Encoding:** Assigns ordered integer values; needed for ordinal data (Low, Medium, High).
  Example:  
  `mapping = {"Low": 0, "Medium": 1, "High": 2}`  
  | **Original** | **Encoded** |  
  |--------------|-------------|  
  | "Low"        | 0           |  
  | "High"       | 2           | 


In [24]:
#Label Encoding:

from sklearn.preprocessing import LabelEncoder

df = pd.DataFrame({'grade': ['A', 'B', 'A', 'C']})
le = LabelEncoder()
df['grade_encoded'] = le.fit_transform(df['grade'])
print(df)


  grade  grade_encoded
0     A              0
1     B              1
2     A              0
3     C              2


In [26]:
#One-Hot Encoding:

df = pd.DataFrame({'color': ['Red', 'Blue', 'Green']})
df_encoded = pd.get_dummies(df, columns=['color'])
print(df_encoded)

   color_Blue  color_Green  color_Red
0       False        False       True
1        True        False      False
2       False         True      False


In [67]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
import pandas as pd

# Sample data
data = pd.DataFrame({
    "City": ["Paris", "Tokyo", "Paris", "New York"],  # Nominal
    "Size": ["S", "M", "L", "S"],                    # Ordinal
})

# One-Hot Encoding for nominal data
onehot = OneHotEncoder(sparse_output=False)
city_encoded = onehot.fit_transform(data[["City"]])
print(city_encoded)

# Ordinal Encoding for ordinal data
ordinal_mapping = [["S", "M", "L"]]  # Define order: S < M < L
ordinal = OrdinalEncoder(categories=ordinal_mapping)
size_encoded = ordinal.fit_transform(data[["Size"]])
print(size_encoded)

[[0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
[[0.]
 [1.]
 [2.]
 [0.]]


## 3. Encoding Numerical Data

**Explanation:**  
Binning or discretizing numerical features to turn continuous variables into categories.

**Common Techniques:**
- **Binning:** Groups values into discrete intervals or categories.

**When to Use:**
- When variables are highly skewed, or model interpretability improves with categories.
- Useful in tree-based models and when capturing non-linear relationships or thresholds.


In [28]:
df = pd.DataFrame({'age': [22, 35, 58, 44]})
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 100], labels=['Young', 'Middle', 'Senior'])
print(df)

   age age_group
0   22     Young
1   35    Middle
2   58    Senior
3   44    Middle


---
### **Good to know Section:**
#### **Other Advanced Techniques for High-Cardinality/Complex Data**
When categories have **many unique values** (e.g., ZIP codes, product IDs), or complex relationships to the target variable:  

**1. Frequency Encoding**
- **Mechanism**: Replaces categories with their occurrence frequency.  
  Example:  
  "City" frequencies: {"Tokyo": 0.25, "Paris": 0.15}  
- **Use Case**:  
  - High-cardinality features where frequency correlates with target.  
- **Pros**:  
  - No dimensionality increase.  
- **Cons**:  
  - Loses uniqueness; different categories with same frequency get identical values.  

**2. Target Encoding (Mean Encoding)**
- **Mechanism**: Replaces categories with the **mean of the target variable** for that category.  
  Example:  
  For binary classification, "City" becomes:  
  `Mean(Target | City = "Tokyo")`  
- **Use Case**:  
  - High-cardinality features with predictive power.  
- **Pros**:  
  - Captures complex relationships to target.  
- **Cons**:  
  - Risk of **overfitting** (always use cross-validation).  
  - Requires regularization (e.g., smoothing).  

**3. Hashing Encoding**
- **Mechanism**: Uses a hash function to map categories to fixed dimensions.  
  Example:  
  100 categories → hashed into 10 columns.  
- **Use Case**:  
  - Extremely high-cardinality features (e.g., user IDs).  
- **Pros**:  
  - Fixed dimensionality regardless of categories.  
- **Cons**:  
  - Irreversible (can't trace back categories).  
  - Potential hash collisions.

---
**When to Use Which Technique?**
| **Scenario**                           | **Recommended Technique**       |
|----------------------------------------|----------------------------------|
| **Ordinal data** (ordered categories)  | `Label Encoding` or `Ordinal Encoding` |
| **Nominal data** (unordered categories)| `One-Hot Encoding`               |
| **High-cardinality features**          | `Frequency`/`Target`/`Hashing Encoding` |
| **Tree-based models** (XGBoost, etc.)  | `Label Encoding` often suffices  |
| **Linear models** (regression, SVM)    | `One-Hot Encoding` preferred     |

---

**Key Considerations**
1. **Avoid Data Leakage**:  
   - Fit encoders **only on training data** (e.g., `LabelEncoder.fit(X_train)`).  
2. **Handling Unseen Categories**:  
   - Define strategies for test-set categories not seen during training (e.g., treat as "unknown").  
3. **Sparsity vs. Dimensionality Trade-off**:  
   - For >20 categories, prefer `Target Encoding` or `Hashing` over `One-Hot`.  
4. **Tree-Based Models**:  
   - May work with label encoding even for nominal data (splits ignore false order).

---

**Key Pointers for Encoding**
- **Nominal Data → One-Hot Encoding** (prevents false ordering).  
- **Ordinal Data → Label/Ordinal Encoding** (preserves meaningful order).  
- **High Cardinality → Target/Frequency/Hashing Encoding** (avoids dimensionality explosion).  
Always validate encoding choices by monitoring model performance!

## 4. Column Transformers

**Explanation:**  
Facilitates the application of different preprocessing steps to specific columns within a dataset (e.g., scaling numerics, encoding categoricals). Real-world datasets contain mixed data types (numerical, categorical) requiring distinct transformations (e.g., scaling, encoding). Column transformers handle these efficiently in a single ste

**Key Concepts**  
- **Column Selection**:  
  Specify columns by name, index, or dtype (e.g., `numerical_columns`, `categorical_columns`).  
- **Transformers**:  
  Apply any scikit-learn-style transformer (e.g., `StandardScaler`, `OneHotEncoder`, `PowerTransformer`).  
- **Parallel Execution**:  
  All transformations are applied **simultaneously** (no sequential dependency), optimizing performance.  
- **Sparse Handling**:  
  Intelligently merges sparse (e.g., one-hot) and dense outputs.

**When to Use:**
- Essential for datasets containing mixed data types that require separate transformations.
- Ensures all transformations are applied consistently within cross-validation or integrated workflows[4][5].

##### **Advantages**  
- **Avoid Data Leakage**:  
  All transformations encapsulated in one object.  
- **Code Simplicity**:  
  Eliminate manual column splitting/recombining.  
- **Reproducibility**:  
  Deploy the same preprocessing to new data.  

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.DataFrame({
    'city': ['Delhi', 'Mumbai', 'Chennai'],
    'salary': [70000, 120000, 90000]
})

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), ['city']),
        ('num', StandardScaler(), ['salary'])
    ]
)

X_transformed = preprocessor.fit_transform(data)
print(X_transformed)


[[ 0.          1.          0.         -1.13554995]
 [ 0.          0.          1.          1.29777137]
 [ 1.          0.          0.         -0.16222142]]


## 5. Pipelines

**Explanation:**  
Automates and standardizes the sequence of preprocessing and modeling steps, linking transformations and models into a single workflow.

**When to Use:**
- For reproducibility, clarity, and automation.
- When you need to ensure consistent transformation during training and testing, or when scaling up to production or team settings[6][7].



In [32]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler())
])

data = [[1, 1000], [2, 2000]]
pipeline.fit_transform(data)


array([[-1., -1.],
       [ 1.,  1.]])

## 6. Function Transformers

**Explanation:**  
Applies custom or predefined mathematical functions (like log, square root) to features.

**When to Use:**
- When addressing skewness, variance instability, or specific domain-driven transformations.
- Useful for numeric features that need a particular transformation for better model fit[8][9].

**Key Characteristics**:
- **Flexibility**: Any Python function (log, sqrt, custom math)
- **Stateless Operations**: No learned parameters (usually)
- **Pipeline Integration**: Works with cross-validation


**Use Cases**:
- Custom transformations (log, box-cox without parameter search)
- Feature engineering (combinations, interactions)
- Data cleaning operations
- Wrapping non-scikit-learn functions

**Limitations**:
- No parameter learning from data
- Requires vectorized operations for efficiency


In [34]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

log_transformer = FunctionTransformer(np.log1p, validate=True)
data = [[1], [10], [100]]
print(log_transformer.fit_transform(data))


[[0.69314718]
 [2.39789527]
 [4.61512052]]


## 7. Power Transformers

**Explanation:**  
Power transformers are techniques used to modify non-normally distributed data to resemble a Gaussian (normal) distribution. This is critical for many statistical models and machine learning algorithms that assume normality (e.g., linear regression, Gaussian processes). It transforms data to approximate a normal (Gaussian) distribution using techniques like Box-Cox (only works for strictly positive data) and Yeo-Johnson (similar to Box-Cox and works for all real numbers {positive, negative, and zero}).

### **Why Transform Data to Normal Distribution?**
1. **Statistical Assumptions**:  
   Many parametric tests (e.g., t-tests, ANOVA) and models require normally distributed data for validity.
2. **Model Performance**:  
   Algorithms like linear regression perform better when residuals are normally distributed.
3. **Stabilize Variance**:  
   Reduces heteroscedasticity (uneven variance across data ranges).
4. **Improve Interpretability**:  
   Normal distributions simplify analysis through mean and standard deviation.

---

### **Practical Considerations**
- **Inverse Transformation**:  
  Essential to revert data to original scale after prediction (e.g., in regression).  
- **Impact on Models**:  
  - **Linear Models**: Often improves accuracy and interpretability.  
  - **Tree-Based Models** (e.g., Random Forest): Less critical (invariant to monotonic transforms).  
- **Data Leakage**:  
  Compute \(\lambda\) on **training data only**, then apply to test data.  
- **Standardization**:  
  Often combined with scaling (e.g., mean=0, std=1) post-transformation.  

---

### **When to Use Power Transformers?**
- **Skewed Data**: Right/left-skewed distributions (e.g., income, sensor readings).  
- **Heteroscedasticity**: When variance changes across data ranges.  
- **Model Requirements**: For algorithms demanding normality (e.g., LDA, ARIMA).
- Box-Cox is suited for positive data, Yeo-Johnson for data that includes negatives[10][9].

### **When to Avoid?**
- **Categorical Data**: Only for continuous numeric features.  
- **Near-Normal Data**: Unnecessary overhead if data is already normal.  
- **Tree-Based Models**: Minimal impact on performance.  

---


In [36]:
from sklearn.preprocessing import PowerTransformer

data = [[1, 10], [2, 100], [3, 1000]]
pt = PowerTransformer()
data_trans = pt.fit_transform(data)
print(data_trans)


[[-1.25218937 -1.22196871]
 [ 0.05687037 -0.00553358]
 [ 1.19531901  1.22750228]]


In [53]:
from sklearn.preprocessing import PowerTransformer
import numpy as np

# Sample data (with negatives/zeros)
data = np.array([-2, 0, 1, 4, 10]).reshape(-1, 1)

# Yeo-Johnson (handles all values)
pt = PowerTransformer(method='yeo-johnson', standardize=True)
transformed = pt.fit_transform(data)

# Inverse transform
original = pt.inverse_transform(transformed)


**Power transformers** (Box-Cox for positive data, Yeo-Johnson for any data) are essential tools to stabilize variance and approximate normality. By optimizing \(\lambda\), they enhance model performance and statistical validity. Always validate results visually/statistically and handle data leakage carefully during implementation.

#### **Comprehensive Guide to Column Transformers, Function Transformers, and Power Transformers**

**Key Differences & Selection Guide**
| **Transformer**       | **Primary Use Case**                     | **Learn Parameters?** | **Handles Mixed Data?** |
|-----------------------|------------------------------------------|-----------------------|-------------------------|
| **Column Transformer**| Different processing per feature type    | Yes (per column set)  | ✅                      |
| **Function Transformer**| Custom operations/feature engineering | ❌ (stateless)         | ❌                      |
| **Power Transformer** | Normalize feature distributions          | ✅ (λ optimization)    | ❌                      |

---

## 8. Handling Mixed Data

**Explanation:**  
Integrates multiple types of features (numerical, categorical, ordinal) and applies the correct transformation to each type.

**When to Use:**
- Whenever data comprises both categorical and numerical features.
- Enhances preprocessing robustness by ensuring each type is handled optimally[5].

In [38]:
mixed_data = pd.DataFrame({
    'city': ['NY', 'LA', 'SF'],
    'income': [70000, 50000, 80000]
})

ct = ColumnTransformer(
    [('cats', OneHotEncoder(), ['city']),
     ('nums', StandardScaler(), ['income'])]
)
mixed_transformed = ct.fit_transform(mixed_data)
print(mixed_transformed)


[[ 0.          1.          0.          0.26726124]
 [ 1.          0.          0.         -1.33630621]
 [ 0.          0.          1.          1.06904497]]


## 9. Handling Date-Time Features

**Explanation:**  
Extracts useful information (like year, month, weekday, hour) or converts dates into a numeric format for modeling.

**When to Use:**
- When temporal patterns or trends are relevant (retail, finance, event-based data).
- For creating features that help models capture seasonality, holidays, or time-dependent effects.



In [40]:
df = pd.DataFrame({'date': ['2021-07-21', '2022-12-15']})
df['date'] = pd.to_datetime(df['date'])
df['weekday'] = df['date'].dt.dayofweek
print(df)


        date  weekday
0 2021-07-21        2
1 2022-12-15        3


## 10. Handling Missing Values

**Explanation:**  
Deals with incomplete data through imputation or removal.

**Common Solutions:**
- **Mean/Median/Mode Imputation:** For numerical/categorical data.
- **Predictive Imputation or Interpolation:** For more complex gaps.

**When to Use:**
- Whenever datasets contain missing entries.
- Critical to maintain data integrity; imputed values can prevent model bias or loss of information[11].

In [42]:
df = pd.DataFrame({'salary': [50000, None, 80000]})
df['salary_filled'] = df['salary'].fillna(df['salary'].mean())
print(df)

    salary  salary_filled
0  50000.0        50000.0
1      NaN        65000.0
2  80000.0        80000.0


In [44]:
from sklearn.impute import SimpleImputer

data = [[1, None], [2, 2], [None, 6]]
imp = SimpleImputer(strategy='mean')
imputed = imp.fit_transform(data)
print(imputed)


[[1.  4. ]
 [2.  2. ]
 [1.5 6. ]]


## 11. Handling Outliers

**Explanation:**  
Detects and mitigates extreme values that can skew model training.

**Common Techniques:**
- **Capping/Winsorization:** Limits extreme values to certain percentiles.
- **Z-score or IQR Filtering:** Identifies outliers for exclusion or treatment.

**When to Use:**
- When outliers may unduly influence models (especially those sensitive to distribution or mean/variance, like linear regression).
- Decision depends on business impact—sometimes outliers hold valuable information[9].



In [46]:
df = pd.DataFrame({'income': [30000, 50000, 120000, 900000]})
df['income_capped'] = df['income'].clip(lower=30000, upper=150000)
print(df)

   income  income_capped
0   30000          30000
1   50000          50000
2  120000         120000
3  900000         150000


In [50]:
from scipy import stats
import numpy as np

data = np.array([10, 12, 15, 16, 14, 400])  # 400 is an outlier
z = np.abs(stats.zscore(data))
filtered_entries = (z < 2)
print(data[filtered_entries])

[10 12 15 16 14]


## Summary Table: When to Use Each Technique

| Transformation          | When to Use                                                             |
|-------------------------|-------------------------------------------------------------------------|
| Feature Scaling         | Datasets with varying numerical ranges; models sensitive to scale        |
| Encoding Categorical    | Any presence of non-numeric features                                    |
| Encoding Numerical      | Skewed numerics, threshold effects, model interpretability              |
| Column Transformers     | Mixed data types needing different preprocessing                        |
| Pipelines               | Automate, reproduce, and scale workflows                               |
| Function Transformers   | Custom/nonlinear transformations, handling data skew                    |
| Power Transformers      | Making data more Gaussian, handling skewness/kurtosis                   |
| Handling Mixed Data     | Datasets with both categorical and numeric features                     |
| Date-Time Handling      | Time-dependent patterns or seasonality                                  |
| Handling Missing Values | Incomplete datasets                                                     |
| Handling Outliers       | Datasets with potentially erroneous/extreme entries                     |

Each technique should be chosen based on both the characteristics of the data and the intended model, always considering the impact on downstream analytics and predictive power[10][1][4][5].

[1] https://www.appliedaicourse.com/blog/feature-scaling-in-machine-learning/
[2] https://datasciencedojo.com/blog/categorical-data-encoding/
[3] https://www.geeksforgeeks.org/machine-learning/categorical-data-encoding-techniques-in-machine-learning/
[4] https://www.sktime.net/en/v0.28.0/api_reference/auto_generated/sktime.transformations.panel.compose.ColumnTransformer.html
[5] https://bait509-ubc.github.io/BAIT509/lectures/lecture5.html
[6] https://www.datarobot.com/blog/what-a-machine-learning-pipeline-is-and-why-its-important/
[7] https://www.geeksforgeeks.org/machine-learning-pipeline/
[8] https://pub.towardsai.net/feature-transformation-3306c7ac9dd5
[9] https://www.linkedin.com/pulse/feature-transformation-techniques-zuhaib-ashraf
[10] https://www.geeksforgeeks.org/machine-learning/feature-transformation-techniques-in-machine-learning/
[11] https://experienceleague.adobe.com/en/docs/experience-platform/query/advanced-statistics/feature-transformation
[12] https://brainalystacademy.com/feature-transformation/
[13] https://tvsnext.com/blog/getting-started-with-feature-transformation-for-machine-learning/
[14] https://www.kaggle.com/code/nargisbegum82/all-about-feature-transformation
[15] https://aws.amazon.com/what-is/transformers-in-artificial-intelligence/
[16] https://na.noark-electric.com/the-role-of-power-transfomers-in-the-electrical-grid/
[17] https://www.ibm.com/think/topics/feature-engineering
[18] https://www.nb-data.com/p/is-it-necessary-for-feature-scaling
[19] https://www.ai21.com/knowledge/tranformer-model/
[20] https://sathee.iitk.ac.in/article/physics/physics-uses-of-transformer/